In [ ]:
import json
import pkgutil
import tempfile
from urllib.parse import quote, unquote, urljoin, urlparse, urlsplit, urlunparse

import geopandas as gpd
import pooch
import py7zr

import open_geodata as geo

In [ ]:
# class DB:
#     def __init__(self, db='general', project='open_geodata') -> None:
#         self.project = project
#         self.cache = None

#         json_data = pkgutil.get_data(
#             package=self.project, resource=f'db/{db}.json'
#         )
#         if isinstance(json_data, bytes):
#             self.json_raw = json.loads(json_data)
#         else:
#             raise Exception('Erro!')
#         self.json = self._flatten()
#         # self._check_keys()

#     def _flatten(self):
#         flat = {}
#         for outer_key, inner_dict in self.json_raw.items():
#             for inner_key, value in inner_dict.items():
#                 new_key = f"{outer_key}.{inner_key}"
#                 flat[new_key] = value
#         return flat

#     def _check_keys(self):
#         for dict_data in self.json.values():
#             if all(key in dict_data for key in ['url', 'hash']):
#                 pass
#             else:
#                 raise Exception('Falta chaves')

#     @property
#     def list_data(self):
#         return list(self.json.keys())

#     def get_base_url(
#         self,
#         name,
#         # github_fix=True,
#         # github_set_version=True,
#         # github_branch='main',
#     ):
#         if name not in self.list_data:
#             raise Exception('Nome Inválido')

#         #
#         url = self.json[name]['url']
#         scheme = urlparse(url=url).scheme
#         netloc = urlparse(url=url).netloc
#         path = Path(urlparse(url=url).path).parent.as_posix()
#         params = urlparse(url=url).params
#         query = urlparse(url=url).query
#         fragment = urlparse(url=url).fragment

#         url = urlunparse((scheme, netloc, path, params, query, fragment))
#         url = quote(url, safe=':/')
#         return url

#     def _get_hash(self, name):
#         if name not in self.list_data:
#             raise Exception('Nome Inválido')

#         return self.json[name]['hash']

#     def _get_filename(self, name):
#         if name not in self.list_data:
#             raise Exception('Nome Inválido')

#         #
#         url = self.json[name]['url']
#         path = Path(urlparse(url=url).path)
#         return unquote(path.name)

#     def get_registry(self, name):
#         return {self._get_filename(name=name): self._get_hash(name=name)}

#     def _create_cache_for_data(self, name):
#         self.cache = pooch.create(
#             path=pooch.os_cache(project=self.project),
#             base_url=self.get_base_url(name=name),
#             # version='v1.8.2',
#             # version_dev='main',
#             registry=db.get_registry(name=name),
#         )
#         return self.cache

#     def get_data(self, name):
#         self.cache = self._create_cache_for_data(name=name)
#         return self.cache.fetch(fname=self._get_filename(name=name))

#     def get_filehash_sha256(self, name):
#         filename = self.get_data(name=name)
#         return pooch.file_hash(filename, alg="sha256")


# db = DB(db='sp_bh_pcj-2020-2035')
# db.json

In [ ]:
layer = 'sp_tjsp.divadmin'
layer = 'sp_bh_pcj-2020-2035.poligonos'
#layer = 'tab.municipio_nome'
# layer = 'sp_bh_pcj-2020-2035.biomas'

In [ ]:
db = geo.data.DB(db='sp')
type(db.json)

In [ ]:
db.list_data

In [ ]:
url = db.get_base_url(name=layer)
url

In [ ]:
db.get_filehash_sha256(name=layer)

In [ ]:
db.list_data

In [ ]:
db._get_filename(name=layer)

In [ ]:
db.get_registry(name=layer)

In [ ]:
db.get_filehash_sha256(name=layer)

In [ ]:
aaa = db._create_cache_for_data(name=layer)
aaa

In [ ]:
list(aaa.path.absolute().glob('*.7z'))

In [ ]:
filepath = db.get_data(name=layer)
filepath

In [ ]:
#aaa = Path(filepath).with_suffix('.7z')
aaa

In [ ]:
filepath

In [ ]:
# gpd.read_file(aaa)


def read_7z_file(file_path_7z):
    file_path_7z = Path(file_path_7z)
    if file_path_7z.is_file():
        with py7zr.SevenZipFile(file_path_7z, 'r') as archive:
            allfiles = archive.getnames()
            # Quero apenas um arquivo por gpkg
            if len(allfiles) == 1:
                with tempfile.TemporaryDirectory() as temp_dir:
                    temp_dir = Path(temp_dir)
                    archive.extract(path=temp_dir, targets=allfiles)
                    filename = list(temp_dir.glob('*'))[0]
                    ext = filename.suffix.lower()
                    print(ext)
                    if ext in ['.gpkg']:
                        return gpd.read_file(filename=filename)

            else:
                raise RuntimeError('.7z tem mais de um arquivo')


read_7z_file(filepath)

<br>

---

## Padrão do JSON


```json
{
  "sp_bh_pcj-2020-2035": {
    "land": {
      "url": "https://github.com/open-geodata/sp_bh_pcj-2020-2035/blob/main/sp_bh_pcj_2020_2035/data/output/geo/rm%20piracicaba%20-%20poligonos.7z",
      "attribution": "Comitês das Bacias PCJ",
      "name": "geo.rm_piracicaba",
      "description": "Alguma coisa que eu lembro",
      "details": "https://github.com/open-geodata/sp_bh_pcj-2020-2035",
      "hash": "1926c621afd6ac67c3f36639bb1236134a48d82226dc675d3e3df53d02d2a3de",
      "filename": "ne_110m_land.zip"
    }
  }
}
```

```json
{
  "sp_bh_pcj-2020-2035.land": {
    "url": "https://github.com/open-geodata/sp_bh_pcj-2020-2035/blob/main/sp_bh_pcj_2020_2035/data/output/geo/rm%20piracicaba%20-%20poligonos.7z",
    "attribution": "Comitês das Bacias PCJ",
    "name": "geo.rm_piracicaba",
    "description": "Alguma coisa que eu lembro",
    "details": "https://github.com/open-geodata/sp_bh_pcj-2020-2035",
    "hash": "1926c621afd6ac67c3f36639bb1236134a48d82226dc675d3e3df53d02d2a3de",
    "filename": "ne_110m_land.zip"
  }
}
```
